## NBA Broadcast Slot Scheduling - CSP/COP with CP-SAT

**Module**: Constraint Satisfaction / Optimization Problem  
**Technique**: CP-SAT (Google OR-Tools) - CDCL + constraint propagation  
**Problem**: Assigning NBA games to television broadcast slots while respecting operational and business constraints and maximizing exposure for high-profile games in prime time.

**Notebook structure:**
1. Setup & Environment
2. Input Data (demonstration data structured around the real NBA domain)
3. Model Initialization - binary variables x_{m,s}
4. Hard Constraints - assignment, slot capacity, team conflict
5. Soft/Business Constraints - prime-time cap, big-match distribution
6. Objective Function - maximizing prime-time exposure
7. Solver Execution
8. Output & Visualization
9. Knowledge Base Analysis - variables, constraints, complexity

**Data note**: The model operates on a demonstration instance (12 games, 14 slots, 12 teams). The constraint structure is designed to scale directly to real instances (1,230 games per season, 30 NBA teams).

### 0. Motivation and Real-World Context

#### The Real Problem

Each NBA season assigns roughly **1,230 games** to television broadcast slots on national channels (ESPN, TNT, ABC, NBA TV). The assignment must respect operational constraints (each game appears exactly once, each slot hosts at most one game, no team plays twice on the same day) and business constraints (high-profile games should be placed in prime time, major rivalries should not overlap on the same channel).

The real dataset used in the ML module of this project (`features_nba_data_2000-01_2025-26.csv`, 31,160 games, seasons 2000-01 -> 2025-26) includes the features `home_is_back_to_back` and `away_is_back_to_back`, which record whether a team played the previous night. Across the full dataset, **36.4%** of games involve at least one team in a back-to-back situation, an effect directly produced by scheduling decisions.

This CSP model is a demonstration prototype that formalizes the same scheduling logic on a reduced instance (12 games, 14 slots), using the same constraint structure that would apply at real scale.

#### Link with the Other Project Modules

| Module | Technique | Input | Output |
|--------|-----------|-------|--------|
| ML (notebook 06) | Random Forest / XGBoost / GBM | Rolling game features | Outcome prediction (`home_win`, `point_diff`) |
| **CSP (this notebook)** | **CP-SAT (OR-Tools)** | **Games and available slots** | **Optimal schedule** |
| A* (`road_trip_optimizer`) | A* search + MST heuristic | Arenas and logistics costs | Optimal road trip |

The ML model and the CSP operate on different layers: ML **predicts** outcomes given a schedule, while the CSP **constructs** a schedule that satisfies constraints. In an integrated system, the CSP output would feed the ML model (which schedule maximizes expected audience?) and the A* output would inform logistics costs in the scheduling constraints (for example, limiting back-to-backs caused by long road trips).

### 1. Setup & Environment

We use `pandas` for tabular input/output inspection and OR-Tools CP-SAT for the binary optimization model.

The core solver import is:

```python
from ortools.sat.python import cp_model
```

CP-SAT is appropriate here because each game-slot assignment is naturally represented as a binary decision variable.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, asdict
from typing import Dict, List, Sequence, Tuple

import pandas as pd
from ortools.sat.python import cp_model

@dataclass(frozen=True)
class Team:
    """NBA team metadata used by the scheduling model."""

    team_id: str
    name: str
    conference: str


@dataclass(frozen=True)
class Game:
    """Game demand that must be assigned to exactly one broadcast slot."""

    game_id: str
    home_team: str
    away_team: str
    is_big_match: bool


@dataclass(frozen=True)
class Slot:
    """Available TV inventory for a specific channel, day, and broadcast window."""

    slot_id: str
    day: str
    start_time: str
    channel: str
    is_prime_time: bool


AssignmentVars = Dict[Tuple[str, str], cp_model.IntVar]

### 2. Input Data Mockup

The model consumes three data structures:

- `Teams`: team metadata used for team-level conflict constraints.
- `Games`: game demand, including `is_big_match` to identify marquee games.
- `Slots`: TV inventory, including `is_prime_time`, `channel`, `day`, and `start_time`.

The extra `start_time` field supports the business rule that no two big matches should air on the same channel simultaneously.

In [ ]:
def create_mock_data() -> Tuple[List[Team], List[Game], List[Slot]]:
    """Create realistic, compact input data for model demonstration.

    Returns:
        A tuple containing teams, games, and available TV slots.
    """

    teams = [
        Team("BOS", "Boston Celtics", "East"),
        Team("NYK", "New York Knicks", "East"),
        Team("CHI", "Chicago Bulls", "East"),
        Team("PHI", "Philadelphia 76ers", "East"),
        Team("LAL", "Los Angeles Lakers", "West"),
        Team("GSW", "Golden State Warriors", "West"),
        Team("DEN", "Denver Nuggets", "West"),
        Team("MIA", "Miami Heat", "East"),
        Team("DAL", "Dallas Mavericks", "West"),
        Team("PHX", "Phoenix Suns", "West"),
        Team("MEM", "Memphis Grizzlies", "West"),
        Team("SAS", "San Antonio Spurs", "West"),
    ]

    games = [
        Game("G1", "BOS", "NYK", True),
        Game("G2", "LAL", "GSW", True),
        Game("G3", "DEN", "MIA", True),
        Game("G4", "DAL", "PHX", False),
        Game("G5", "CHI", "PHI", False),
        Game("G6", "MEM", "SAS", False),
        Game("G7", "BOS", "LAL", True),
        Game("G8", "GSW", "DEN", True),
        Game("G9", "MIA", "DAL", False),
        Game("G10", "PHX", "MEM", False),
        Game("G11", "PHI", "CHI", False),
        Game("G12", "SAS", "NYK", False),
    ]

    slots = [
        Slot("S1", "Monday", "19:00", "ESPN", False),
        Slot("S2", "Monday", "21:30", "TNT", True),
        Slot("S3", "Tuesday", "19:00", "ESPN", False),
        Slot("S4", "Tuesday", "21:30", "TNT", True),
        Slot("S5", "Wednesday", "19:00", "ESPN", False),
        Slot("S6", "Wednesday", "21:30", "TNT", True),
        Slot("S7", "Thursday", "19:00", "NBA TV", False),
        Slot("S8", "Thursday", "21:30", "ESPN", True),
        Slot("S9", "Friday", "19:00", "NBA TV", False),
        Slot("S10", "Friday", "21:30", "ESPN", True),
        Slot("S11", "Saturday", "19:00", "TNT", True),
        Slot("S12", "Saturday", "21:30", "ABC", True),
        Slot("S13", "Sunday", "19:00", "TNT", True),
        Slot("S14", "Sunday", "21:30", "ABC", True),
    ]

    return teams, games, slots


teams, games, slots = create_mock_data()

pd.DataFrame([asdict(game) for game in games])

In [ ]:
pd.DataFrame([asdict(slot) for slot in slots])

### 3. Model Initialization

Let \(M\) be the set of games and \(S\) be the set of broadcast slots.

The binary decision variable is:

$$
x_{m,s} =
\begin{cases}
1, & \text{if game } m \text{ is assigned to slot } s \\
0, & \text{otherwise}
\end{cases}
$$

for every \(m \in M\) and \(s \in S\).

In [ ]:
def initialize_model(
    games: Sequence[Game], slots: Sequence[Slot]
) -> Tuple[cp_model.CpModel, AssignmentVars]:
    model = cp_model.CpModel()
    x = {
        (game.game_id, slot.slot_id): model.NewBoolVar(
            f"x_{game.game_id}_{slot.slot_id}"
        )
        for game in games
        for slot in slots
    }
    return model, x

### 4. Hard Constraints

#### Assignment Constraint

Each game must be assigned to exactly one broadcast slot:

$$
\sum_{s \in S} x_{m,s} = 1 \quad \forall m \in M
$$

#### Slot Resource Constraint

Each slot can host at most one game:

$$
\sum_{m \in M} x_{m,s} \leq 1 \quad \forall s \in S
$$

#### Team Conflict Constraint

A team cannot play more than one game on the same day. Let \(M_t\) be the games involving team \(t\), and \(S_d\) be the slots on day \(d\):

$$
\sum_{m \in M_t} \sum_{s \in S_d} x_{m,s} \leq 1 \quad \forall t \in T, d \in D
$$

In [ ]:
def add_hard_constraints(
    model: cp_model.CpModel,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
    teams: Sequence[Team],
) -> None:
    # Complete schedule: every game appears exactly once.
    for game in games:
        model.Add(sum(x[(game.game_id, slot.slot_id)] for slot in slots) == 1)

    # Broadcast inventory: each slot has capacity for one game.
    for slot in slots:
        model.Add(sum(x[(game.game_id, slot.slot_id)] for game in games) <= 1)

    # Player/team welfare and operational realism: no team plays twice per day.
    days = sorted({slot.day for slot in slots})
    for team in teams:
        team_games = [
            game for game in games if team.team_id in {game.home_team, game.away_team}
        ]
        for day in days:
            day_slots = [slot for slot in slots if slot.day == day]
            model.Add(
                sum(
                    x[(game.game_id, slot.slot_id)]
                    for game in team_games
                    for slot in day_slots
                )
                <= 1
            )

### 5. Soft / Business Constraints

These rules are implemented as hard constraints in this first version, but they are isolated in `add_business_constraints()` so they can later be relaxed with penalty variables.

#### Prime-Time Capacity

Let \(S^P\) be the set of prime-time slots and \(N\) be the maximum number of games allowed in prime-time inventory:

$$
\sum_{m \in M} \sum_{s \in S^P} x_{m,s} \leq N
$$

#### Big Match Distribution

Let \(M^B\) be the set of big matches. For each broadcast window \((d, c, h)\), where \(d\) is day, \(c\) is channel, and \(h\) is start time, no more than one big match can be scheduled simultaneously on that channel:

$$
\sum_{m \in M^B} \sum_{s \in S_{d,c,h}} x_{m,s} \leq 1
\quad \forall (d,c,h)
$$

In [ ]:
def add_business_constraints(
    model: cp_model.CpModel,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
    max_prime_time_games: int,
) -> None:
    prime_time_slots = [slot for slot in slots if slot.is_prime_time]
    model.Add(
        sum(
            x[(game.game_id, slot.slot_id)]
            for game in games
            for slot in prime_time_slots
        )
        <= max_prime_time_games
    )

    big_games = [game for game in games if game.is_big_match]
    broadcast_windows = sorted(
        {(slot.day, slot.channel, slot.start_time) for slot in slots}
    )
    for day, channel, start_time in broadcast_windows:
        matching_slots = [
            slot
            for slot in slots
            if slot.day == day and slot.channel == channel and slot.start_time == start_time
        ]
        model.Add(
            sum(
                x[(game.game_id, slot.slot_id)]
                for game in big_games
                for slot in matching_slots
            )
            <= 1
        )

### 6. Objective Function

The business objective is to maximize marquee-game exposure by assigning big matches to prime-time slots.

Let \(M^B\) be big matches and \(S^P\) be prime-time slots:

$$
\max \sum_{m \in M^B} \sum_{s \in S^P} x_{m,s}
$$

This objective is intentionally simple and interpretable. Later iterations could add weights by rivalry value, expected audience, travel fairness, or channel priority.

In [ ]:
def add_objective(
    model: cp_model.CpModel,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
) -> None:
    model.Maximize(
        sum(
            x[(game.game_id, slot.slot_id)]
            for game in games
            if game.is_big_match
            for slot in slots
            if slot.is_prime_time
        )
    )

### 7. Solver Execution

We solve the model with `cp_model.CpSolver()` and set `max_time_in_seconds` to bound the search.

The solver can return several statuses. For reporting, we treat `OPTIMAL` and `FEASIBLE` as valid schedules. `INFEASIBLE` triggers diagnostic guidance about which constraints are most likely to require relaxation.

In [ ]:
def solve_model(
    model: cp_model.CpModel, max_time_in_seconds: float = 10.0
) -> Tuple[cp_model.CpSolver, int]:
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = max_time_in_seconds
    status = solver.Solve(model)
    return solver, status

### 8. Output & Visualization

The final schedule is extracted into a tidy `pandas.DataFrame` with one row per assigned game. This makes the output easy to sort, inspect, export, or feed into downstream visualization layers.

If the model is infeasible, the diagnostic block suggests relaxation candidates in a practical order: inventory, prime-time cap, team conflicts, and big-match distribution.

In [ ]:
def extract_schedule(
    solver: cp_model.CpSolver,
    status: int,
    x: AssignmentVars,
    games: Sequence[Game],
    slots: Sequence[Slot],
) -> pd.DataFrame:
    if status not in {cp_model.OPTIMAL, cp_model.FEASIBLE}:
        raise ValueError("Schedule extraction requires a feasible solver status.")

    rows = []
    slot_order = {slot.slot_id: order for order, slot in enumerate(slots)}
    for game in games:
        for slot in slots:
            if solver.BooleanValue(x[(game.game_id, slot.slot_id)]):
                rows.append(
                    {
                        "game_id": game.game_id,
                        "home_team": game.home_team,
                        "away_team": game.away_team,
                        "is_big_match": game.is_big_match,
                        "slot_id": slot.slot_id,
                        "day": slot.day,
                        "start_time": slot.start_time,
                        "channel": slot.channel,
                        "is_prime_time": slot.is_prime_time,
                        "_slot_order": slot_order[slot.slot_id],
                    }
                )

    return (
        pd.DataFrame(rows)
        .sort_values("_slot_order")
        .drop(columns="_slot_order")
        .reset_index(drop=True)
    )


def print_infeasibility_guidance() -> None:
    print("No feasible schedule found.")
    print("Consider relaxing constraints in this order:")
    print("1. Increase available slots or reduce the game set.")
    print("2. Raise max_prime_time_games if business inventory is too tight.")
    print("3. Allow controlled team conflict exceptions with penalty variables.")
    print("4. Relax big-match channel distribution for low-priority windows.")

In [ ]:
def build_model(
    teams: Sequence[Team],
    games: Sequence[Game],
    slots: Sequence[Slot],
    max_prime_time_games: int,
) -> Tuple[cp_model.CpModel, AssignmentVars]:
    model, x = initialize_model(games, slots)
    add_hard_constraints(model, x, games, slots, teams)
    add_business_constraints(model, x, games, slots, max_prime_time_games)
    add_objective(model, x, games, slots)
    return model, x


max_prime_time_games = 7
model, x = build_model(teams, games, slots, max_prime_time_games)
solver, status = solve_model(model, max_time_in_seconds=10.0)

print(f"Solver status: {solver.StatusName(status)}")
if status in {cp_model.OPTIMAL, cp_model.FEASIBLE}:
    print(f"Objective value: {solver.ObjectiveValue():.0f}")
    schedule_df = extract_schedule(solver, status, x, games, slots)
    display(schedule_df)
elif status == cp_model.INFEASIBLE:
    print_infeasibility_guidance()
else:
    print("Search ended before proving feasibility or infeasibility.")
    print("Try increasing max_time_in_seconds or simplifying the model.")

### 9. Knowledge Base (KB) Analysis of the System

This CSP acts as a **procedural-declarative Knowledge Base** where:
- **Facts** = Team, Game, and Slot instances (input data)
- **Hard rules** = feasibility constraints (assignment, slot capacity, team conflict)
- **Soft rules** = business constraints (prime-time cap, big-match distribution)
- **Objective** = the audience maximization function (prime-time exposure)

#### 9.1 Representation

The decision variables are Boolean variables (BoolVar in CP-SAT):

$$
x_{m,s} \in \{0, 1\} \quad \forall m \in M, \, s \in S
$$

where $M$ is the set of games and $S$ is the set of slots.

The domain of each variable is binary: CP-SAT internally represents each BoolVar as an IntVar with domain $[0, 1]$. The KB is therefore represented as a **binary Constraint Satisfaction Problem** with integer variables over a discrete domain.

#### 9.2 Constraint Families and Their Semantics

In [ ]:
# -- Section 9: Quantitative KB analysis ------------------------------------

import math

# Model parameters extracted from variables already defined in the notebook.
n_games = len(games)
n_slots = len(slots)
n_teams = len(teams)
n_days = len({slot.day for slot in slots})
n_big = sum(1 for game in games if game.is_big_match)
n_prime = sum(1 for slot in slots if slot.is_prime_time)

# Number of decision variables.
n_vars = n_games * n_slots

# Number of constraints by family.
c_assignment = n_games          # One constraint per game: sum == 1.
c_slot_capacity = n_slots       # One constraint per slot: sum <= 1.
c_team_conflict = n_teams * n_days  # One constraint per (team, day).

# Business constraints.
n_broadcast_windows = len({(slot.day, slot.channel, slot.start_time) for slot in slots})
c_prime_time_cap = 1                 # One global constraint.
c_big_match_dist = n_broadcast_windows  # One constraint per broadcast window.

c_total = (
    c_assignment
    + c_slot_capacity
    + c_team_conflict
    + c_prime_time_cap
    + c_big_match_dist
)

# Naive solution space without constraints.
# Each variable can be 0 or 1, so there are 2^(n_games * n_slots) configurations.
log2_naive = n_vars  # log2(2^n_vars) = n_vars.

print("=" * 60)
print("Quantitative Knowledge Base Analysis")
print("=" * 60)
print(f"\nDecision variables (BoolVar):      {n_vars:>6d}")
print(f"  - Games x slots:                {n_games:>3d} x {n_slots:>3d}")
print()
print(f"Total constraints:                 {c_total:>6d}")
print(f"  - Assignment (hard):            {c_assignment:>6d}  [sum x_{{m,s}} = 1 for all m]")
print(f"  - Slot capacity (hard):         {c_slot_capacity:>6d}  [sum x_{{m,s}} <= 1 for all s]")
print(f"  - Team conflict (hard):         {c_team_conflict:>6d}  [no double-booking for all (t,d)]")
print(f"  - Prime-time cap (business):    {c_prime_time_cap:>6d}  [sum prime <= N]")
print(f"  - Big-match dist (business):    {c_big_match_dist:>6d}  [no overlap for all (d,c,h)]")
print()
print(f"Naive solution space (log2):       2^{log2_naive} ~= 10^{log2_naive * math.log10(2):.0f}")
print()

# Theoretical problem complexity.
print("Theoretical complexity:")
print("  The problem is NP-hard in general (resource-constrained scheduling).")
print("  CP-SAT solves it through CDCL (Conflict-Driven Clause Learning) + propagation.")
print(f"  For instances of this size ({n_games} games, {n_slots} slots),")
print("  solving is instantaneous because hard-constraint propagation")
print("  drastically reduces the effective search space.")
print("=" * 60)

#### 9.3 KB Usage: Reasoning and Inference

The CP-SAT solver operates through three main mechanisms:

1. **Constraint propagation**: every time a variable is assigned, constraints propagate restrictions to connected variables. For example, assigning game G1 to slot S1 forces `x_{G1,s} = 0` for every other slot and `x_{m,S1} = 0` for every other game.

2. **Backtracking search** (*CDCL*): the solver explores the search tree with backtracking informed by clause learning. When it detects a conflict, it learns a new constraint that prevents the same situation from recurring.

3. **Optimization** (*branch and bound*): while maximizing the objective, the solver maintains an upper bound and prunes branches that cannot improve it.

The reasoning process **cannot be reduced to simple pattern matching over explicitly stored facts**: the optimal solution emerges from the interaction of constraints and cannot be precomputed without search.

#### 9.4 Hard / Soft Separation and Scalability

The separation between `add_hard_constraints()` and `add_business_constraints()` is a deliberate modeling choice: business constraints are currently hard (violation = infeasibility), but the code structure isolates them so they can later be converted into **soft constraints with penalties**:

```python
# Future soft version: penalty variable instead of a hard constraint.
penalty = model.NewIntVar(0, MAX_PENALTY, "prime_time_overflow")
model.Add(sum(...) <= max_prime_time_games + penalty)
model.Minimize(penalty)  # Or added as a negative term in the objective.
```

This extension would turn the CSP into a pure **COP** (Constraint Optimization Problem), where each violation has an explicit cost in the objective function.